# ISOM 835 · Session 9 — Trust, but Verify
**Suffolk University · Sawyer Business School · Fall 2026 · Mon Nov 16 · Prof. Hasan Arslan**

**Tonight in one sentence:** last week we built a model that scores **0.945**. Tonight we find out whether we should **believe** it, and we learn to answer the five questions every boss asks before trusting a model.

**How to use this notebook:** run the cells top to bottom (Shift + Enter). Read the short text above each cell first: it tells you **what the cell does** and **how to read the output**. You do not need to understand every line of code tonight. You do need to understand every *result*. (The first two code cells set things up. **Run both**: the second one installs the library for tonight's receipts.) Your numbers may differ from the slides in the last digit, because library versions differ a little. The story does not change.

### Tonight's map: five questions from your boss

| Step | Your boss asks… | Our tool | Its real name |
|---|---|---|---|
| **0** | "Remind me, what did we build?" | last week's champion | gradient boosting + early stopping |
| **1–3** | **"Is that score real?"** | five exams instead of one, and an exam that looks like the real job | **cross-validation** |
| **4–5** | **"Can we make it better?"** | turn the dials, honestly | **hyperparameter tuning** |
| **6–7** | **"Why did it say that?"** | a receipt for every prediction | **SHAP** |
| **8** | **"How sure is it?"** | let the model say "I'm not sure" | **conformal prediction** |
| **9–10** | **"Does it work for everyone?"** | check each group of guests, then write the label on the box | **slices** and the **model card** |
| **11** | | Your turn: four questions | |

One idea runs through the whole night: **a score is a claim. Check it before you trust it.**

🎮 **Play along:** the [Trust Explorer](https://isom-835.vercel.app/explore/trust/) turns tonight's ideas into four small games on these same hotel bookings. It works on your phone. We play each one **before** the matching step.

In [ ]:
# Environment check — run this cell first. If it fails in Colab: run  !pip install -q -U scikit-learn pandas  then Runtime → Restart session.
import sys, re, sklearn, pandas as pd, numpy as np
need = {'scikit-learn': ('1.6', sklearn.__version__), 'pandas': ('2.2', pd.__version__), 'numpy': ('1.26', np.__version__)}
v = lambda s: tuple(int(x) for x in re.findall(r'\d+', s)[:2])
old = {k: have for k, (want, have) in need.items() if v(have) < v(want)}
assert not old, f'please upgrade {old}: !pip install -q -U ' + ' '.join(old)
print(f'Python {sys.version.split()[0]} ·', ' · '.join(f'{k} {have}' for k, (_, have) in need.items()), '✓')

In [ ]:
# REQUIRED for Steps 6–7 — tonight's two extra libraries: shap (the receipts) and optuna (one optional cell in Step 4). About 20 seconds.
!pip install -q shap optuna

## The business situation (same hotel, new job)

You are still the **revenue manager** for two hotels in Portugal. **About 37% of bookings get canceled.** Last week you built a model that gives every booking one number: **how likely is it to be canceled?**

Now you want to **use** it. You walk into your boss's office and say: *"Our model scores 0.945."* A good boss does not say "great". A good boss asks **five questions**:

1. **"Is that score real?"** Or did the model just get an easy exam?
2. **"Can we make it better?"**
3. **"Why did it say this guest will cancel?"**
4. **"How sure is it?"**
5. **"Does it work for all our guests?"**

Tonight you learn to answer each one **with evidence**. These are also exactly the pieces of **Project Milestone 3**.

The next cell loads the same bookings and rebuilds the same train/test split as the last two weeks, so every score tonight can be compared with last week's.

In [ ]:
import time, warnings
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score
warnings.filterwarnings('ignore')      # hide version notices from the libraries

# Load the bookings (the competition's training data, hosted in our course repo)
URL = 'https://raw.githubusercontent.com/harslan/isom-835/master/public/data/hotel_bookings.csv'
raw = pd.read_csv(URL)
LEAKS = ['reservation_status', 'reservation_status_date', 'assigned_room_type']   # Session 7: these columns are cheating

bookings = raw.sample(30_000, random_state=835)            # the same random 30,000 bookings as the last two weeks
y = bookings['is_canceled']                                 # target: 1 = canceled
X_raw = bookings.drop(columns=['is_canceled'] + LEAKS)      # inputs: everything else, minus the leaks
X_tr_raw, X_te_raw, y_tr, y_te = train_test_split(X_raw, y, test_size=0.2, stratify=y, random_state=835)

def tree_prep(frame):
    # The prep station for trees (same as last week): fill blanks, turn text into code numbers. No scaling, no one-hot.
    numbers = frame.select_dtypes(include='number').columns.tolist()
    text = [c for c in frame.columns if c not in numbers]
    return ColumnTransformer([
        ('numbers', SimpleImputer(strategy='constant', fill_value=-1), numbers),
        ('text', Pipeline([('fill', SimpleImputer(strategy='constant', fill_value='missing')),
                           ('codes', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))]), text),
    ], verbose_feature_names_out=False).set_output(transform='pandas')

prep = tree_prep(X_raw)
X_tr = prep.fit_transform(X_tr_raw)     # learn the dictionary on the training bookings only
X_te = prep.transform(X_te_raw)         # apply the same dictionary to the test bookings

def new_boost(**dials):
    # Last week's recipe: gradient boosting, up to 3,000 small trees, and the coach (early stopping) decides when to stop.
    # Anything you pass in (for example learning_rate=0.05) changes that dial.
    recipe = dict(max_iter=3000, early_stopping=True, validation_fraction=0.15, n_iter_no_change=50, random_state=835)
    return HistGradientBoostingClassifier(**{**recipe, **dials})

print(f'{len(raw):,} bookings in the file · tonight: {len(X_tr):,} to train, {len(X_te):,} held back as the exam · {y_tr.mean():.0%} canceled')

## 0. Last week's champion

Same grading as always: **AUC**. *If I pick one booking that canceled and one that didn't, how often does the model give the canceler the higher score?* **0.5** = a coin flip, **1.0** = perfect.

The next cell rebuilds last week's winner: gradient boosting with early stopping.

In [ ]:
champion = new_boost().fit(X_tr, y_tr)
champion_auc = roc_auc_score(y_te, champion.predict_proba(X_te)[:, 1])
print(f"Last week's champion: {champion.n_iter_} small trees · test AUC {champion_auc:.3f}")

**Stop and notice:** **0.945**, the same number as last week. This is the number you told your boss.

Now the first question.

---
# Question 1 · "Is that score real?"

## 1. One exam is partly luck

Think about a student who takes **one** exam and scores 94.5. Is she a 94.5 student? Maybe. Or maybe the exam happened to ask the questions she had studied. With one exam you **cannot tell**.

Our model is that student. Its 0.945 comes from **one** exam: the 6,000 test bookings that the random split happened to set aside. A different shuffle would have set aside **different** bookings.

🎮 Play it first: in [Station 1 of the Trust Explorer](https://isom-835.vercel.app/explore/trust/#s1), guess where the next score will land, then reshuffle the exam.

Now the same thing in code. The next cell reshuffles the 30,000 bookings **eight times**. Each time it holds back a different 6,000 as the exam, trains the **same model** on the rest, and grades it.

In [ ]:
def one_exam(shuffle_number):
    # A different shuffle: a different 24,000 bookings to train on, a different 6,000 as the exam. Same model every time.
    A_raw, B_raw, y_A, y_B = train_test_split(X_raw, y, test_size=0.2, stratify=y, random_state=shuffle_number)
    station = tree_prep(X_raw)
    model = new_boost().fit(station.fit_transform(A_raw), y_A)
    return roc_auc_score(y_B, model.predict_proba(station.transform(B_raw))[:, 1])

eight_exams = [one_exam(shuffle_number) for shuffle_number in range(1, 9)]

fig, ax = plt.subplots(figsize=(8, 2.6))
ax.scatter(eight_exams, [0] * 8, s=160, color='#5b6bd8', alpha=0.75, zorder=3)
ax.axvline(champion_auc, color='#d64545', ls='--', lw=2)
ax.text(champion_auc, 1.05, f'the score we told the boss: {champion_auc:.3f}  ', color='#d64545', ha='right', va='center')
ax.text(min(eight_exams), -0.6, 'each dot: the same model, a different exam', color='#5b6bd8', ha='left', va='center')
ax.set_yticks([]); ax.set_ylim(-1, 1.6); ax.set_xlabel('test AUC'); ax.set_title('Same model, same data, eight shuffles: eight different scores')
plt.show()

print('the eight scores:', '  '.join(f'{s:.3f}' for s in eight_exams))
print(f'lowest {min(eight_exams):.3f} · highest {max(eight_exams):.3f} · a gap of {max(eight_exams) - min(eight_exams):.3f}')

**How to read it:** each blue dot is the **same model recipe**, graded on a **different** exam. The red dashed line is the score we reported.

**Stop and notice:**
- The scores run from about **0.938 to 0.947**. Nothing about the model changed. Only **which bookings landed in the exam**.
- Our 0.945 sits **near the top** of the dots. We got a slightly **easy exam**. Not a lie, but a little lucky.
- Remember last week's headline: *"boosting beats the forest by 0.003"*. The luck of the shuffle is **three times bigger** than that (a gap of about 0.009). So was last week's win real, or was it luck? We will settle it in Step 2.

**The lesson:** a single test score is **one draw from a range**. An honest report gives the range, not the luckiest draw.

## 2. Five exams instead of one: cross-validation

How do schools fix the "one exam" problem? **More exams.** We do the same, with a neat trick so that no booking is wasted:

1. Cut the **24,000 training bookings** into **5 equal parts** (called **folds**).
2. **Round 1:** train on parts 2, 3, 4, 5. Use part 1 as the exam.
3. **Round 2:** train on parts 1, 3, 4, 5. Use part 2 as the exam.
4. …and so on. **Every part gets one turn as the exam.**
5. Report the **average** of the five scores, and how much they **spread**.

This is called **5-fold cross-validation** (CV for short). Notice what we did **not** touch: the 6,000 test bookings. They stay locked away. Cross-validation happens **inside the training bookings**.

The next cell draws the plan.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 2.8))
for round_number in range(5):
    for part in range(5):
        is_exam = (part == round_number)
        ax.barh(round_number, 0.96, left=part, height=0.7, color='#d64545' if is_exam else '#c9cff2')
        ax.text(part + 0.48, round_number, 'EXAM' if is_exam else 'train', ha='center', va='center',
                color='white' if is_exam else '#3b4590', fontweight='bold' if is_exam else 'normal')
ax.set_yticks(range(5)); ax.set_yticklabels([f'round {r}' for r in range(1, 6)]); ax.invert_yaxis()
ax.set_xticks([p + 0.48 for p in range(5)]); ax.set_xticklabels([f'part {p}' for p in range(1, 6)])
ax.set_title('5-fold cross-validation: every part takes one turn as the exam')
for side in ['top', 'right', 'bottom', 'left']: ax.spines[side].set_visible(False)
ax.tick_params(length=0); plt.show()

In scikit-learn, all five rounds are **one line**: `cross_val_score`. The next cell runs it for last week's two rivals, the **forest** and the **boosted team**, on the **same five exams**. (It trains ten models, so give it about half a minute.)

In [ ]:
five_exams = StratifiedKFold(n_splits=5, shuffle=True, random_state=835)      # 5 parts; "Stratified" = every part has the same 37% of cancelers

forest = RandomForestClassifier(n_estimators=300, min_samples_leaf=5, n_jobs=-1, random_state=835)
forest_scores = cross_val_score(forest, X_tr, y_tr, cv=five_exams, scoring='roc_auc')
boost_scores = cross_val_score(new_boost(), X_tr, y_tr, cv=five_exams, scoring='roc_auc')

report = pd.DataFrame({'random forest': forest_scores, 'boosting': boost_scores}, index=[f'exam {r}' for r in range(1, 6)])
report['boosting wins by'] = report['boosting'] - report['random forest']
report.loc['AVERAGE'] = report.mean()
print(report.round(3).to_string())
print(f"\nboosting, the honest way to say it: {boost_scores.mean():.3f}, give or take {boost_scores.std():.3f}")

**How to read it:** each row is one exam. Both models took the **same** five exams. The last column is boosting's lead on that exam.

**Stop and notice:**
- Boosting's five scores run from about **0.938 to 0.945**. Its honest score is the average: **0.941, give or take 0.003**. A bit **lower** than the 0.945 we told the boss. That is the price of honesty, and it is the number that will hold up.
- **Boosting beats the forest on all 5 exams out of 5.** One win could be luck. Five out of five on the same exams is a **pattern**. Last week's result was real. Now we *know* it instead of hoping it.
- We still have not touched the 6,000 test bookings.

**The habit:** when you compare two models, or two settings, compare them with **cross-validation**, not with one split.

### 🧠 Quick check

A teammate ran 5-fold cross-validation and got: 0.91, 0.93, 0.92, 0.94, 0.90. She writes in her report: *"Our model scores 0.94."* What is wrong, and what should she write?

<details><summary>Click for the answer</summary>

She reported her **luckiest exam**. The honest number is the **average**: (0.91 + 0.93 + 0.92 + 0.94 + 0.90) / 5 = **0.92**, and the scores spread about 0.015 around it. She should write: *"about 0.92, give or take 0.015."*
</details>

## 3. Does the exam look like the real job?

Five exams are better than one. But all five have something in common: they were made by **shuffling**. Bookings from 2015, 2016 and 2017 are mixed together in training **and** in the exam.

Is that how the hotel will use the model? **No.** Next month, the model will score bookings it has **never seen, from a future it has never seen**. It will learn from the **past** and be judged on the **future**.

Think of a **driving test**. Passing in an empty parking lot says little about driving in Boston traffic. **The exam must look like the job.**

So let's give the model the exam that looks like its job:

- **Train** on every booking that arrives in **2015 or 2016**.
- **Exam:** every booking that arrives in **2017**.

To be fair, we also give it a **shuffled** exam with exactly the same number of training and exam bookings.

In [ ]:
def exam(train_rows, exam_rows):
    # Train last week's recipe on some bookings, grade it on others
    station = tree_prep(X_raw)
    model = new_boost().fit(station.fit_transform(X_raw[train_rows]), y[train_rows])
    return roc_auc_score(y[exam_rows], model.predict_proba(station.transform(X_raw[exam_rows]))[:, 1])

past = (X_raw['arrival_date_year'] < 2017).values          # True for bookings that arrive in 2015 or 2016
future = ~past                                              # the 2017 bookings

shuffled = np.zeros(len(y), dtype=bool)                     # a shuffled split of exactly the same size, for a fair comparison
shuffled[np.random.RandomState(835).permutation(len(y))[:past.sum()]] = True

auc_shuffled = exam(shuffled, ~shuffled)
auc_future = exam(past, future)
print(f'train on {past.sum():,} bookings, exam on {future.sum():,}\n')
print(f'shuffled exam (years mixed together):       AUC {auc_shuffled:.3f}')
print(f'the real job (learn 2015–16, predict 2017): AUC {auc_future:.3f}')

twins = X_raw.duplicated(keep=False).mean()
print(f'\n{twins:.0%} of our bookings have an identical twin: another booking with exactly the same values in every column')

**Stop and notice:**
- Same model, same number of bookings. On the **shuffled** exam: about **0.94**. On the exam that looks like the **real job**: about **0.88**. That is a much bigger drop than anything tuning will win back tonight.
- Why? Two reasons:
  1. **Twins.** About **a quarter** of our bookings have an identical twin (think of a tour operator booking 20 identical rooms in one go). When we shuffle, one twin lands in training and the other in the exam. The model has already seen the answer. In real life, next month's bookings are **not sitting in your training data**.
  2. **The world moves.** New travel agents, new prices, a new season. 2017 is not a copy of 2016.
- Neither score is "wrong". They answer **different questions**. The shuffled score answers *"how well does it fill in gaps in data it already knows?"* The past-to-future score answers *"how well will it do **next year**?"* Your boss is asking the second one.

**The lesson, and it is the most important one tonight:** before you choose a model, choose **the exam**. Ask: *"when this model is in use, what will be new to it?"* Then hold out exactly that.

| In real life, what is new? | So hold out… | scikit-learn's name |
|---|---|---|
| future dates | the most recent period | `TimeSeriesSplit` |
| new customers (each customer has many rows) | whole customers | `GroupKFold` |
| nothing special, rows are independent | random parts | `KFold` / `StratifiedKFold` |

For the rest of tonight we keep the **shuffled** exam, so our numbers stay comparable with the last two weeks. In the model card (Step 10) we will tell the truth about it.

### 🧠 Quick check

A bank builds a model that predicts whether a **customer** will miss a payment. The data has 12 rows per customer: one for each month. The team shuffles all rows and gets a wonderful score. Why should you be suspicious?

<details><summary>Click for the answer</summary>

After shuffling, the **same customer** is in training **and** in the exam (some of her months on each side). The model can partly **recognize the customer** instead of learning what makes people miss payments. But in real life the bank will score **new customers**. The fair exam holds out **whole customers** (`GroupKFold`). Expect the score to drop, and trust the lower one.
</details>

---
# Question 2 · "Can we make it better?"

## 4. Turning the dials, honestly

Last week you met the **dials** of a boosted model. You turned two of them by hand in "Your turn":

| Dial | What it controls | Last week's picture |
|---|---|---|
| `learning_rate` | the size of each step | full swings or careful putts |
| `max_leaf_nodes` | how big each small tree is | how many questions each trainee may ask |
| `min_samples_leaf` | the smallest group a tree may make a rule for | "no rules about fewer than 20 guests" |

The real name for a dial is a **hyperparameter**: a setting **you** choose before training. (What the model learns by itself is called a *parameter*.) Finding good settings is called **tuning**.

Think of baking a cake. The dials are **oven temperature**, **baking time** and **amount of sugar**. To find the best cake you bake several and **taste** them. Two questions:

**Which cakes do we bake?**
- **Grid search:** try *every* combination. 5 temperatures × 5 times × 5 sugars = 125 cakes. Thorough, slow.
- **Random search:** bake **12 random** combinations. It sounds lazy, but it usually lands very close to the best, for a tenth of the work.

**Who tastes them?** This is where honesty comes in. **Not the customer.** The test bookings are the customer. We taste with **cross-validation**, inside the training bookings.

The next cell bakes 12 random recipes and tastes each one with 3 exams. That is 36 models, so give it **about a minute**.

In [ ]:
menu = {                                             # the values each dial is allowed to take
    'learning_rate':    [0.03, 0.05, 0.1, 0.2, 0.3],
    'max_leaf_nodes':   [7, 15, 31, 63, 127],
    'min_samples_leaf': [5, 10, 20, 50, 100],
}
start = time.time()
search = RandomizedSearchCV(
    new_boost(), menu,
    n_iter=12,                                                         # try 12 random recipes
    cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=835),    # taste each one with 3 exams (cross-validation)
    scoring='roc_auc', random_state=835, n_jobs=-1)
search.fit(X_tr, y_tr)                                                 # training bookings only: the test set stays locked
print(f'12 recipes × 3 exams = 36 models in {time.time() - start:.0f} seconds\n')

tasting = pd.DataFrame(search.cv_results_['params'])
tasting['CV score'] = search.cv_results_['mean_test_score']
tasting = tasting.sort_values('CV score', ascending=False).reset_index(drop=True)
tasting.index += 1
print(tasting.round(4).to_string())

default_cv = cross_val_score(new_boost(), X_tr, y_tr, cv=search.cv, scoring='roc_auc').mean()
print(f"\nlast week's default dials, tasted the same way: {default_cv:.4f}")
print(f'the best recipe found:                          {search.best_score_:.4f}   {search.best_params_}')

**How to read it:** each row is one recipe (one setting of the three dials). `CV score` is its average over 3 exams. Best recipe on top.

**Stop and notice:**
- The top of the table: **small steps** (0.03 to 0.1) and **medium or big trees** (31 to 127 leaves). The bottom: **tiny trees** (7 leaves) and **big steps** (0.2, 0.3). Just what last week's golf picture said.
- The best recipe beats the default dials by about **0.001**. Honest truth: **tuning is the last few percent, not the first.** Clean data, no leaks and the right exam (Step 3) matter far more.
- We tried 12 recipes and **never looked at the test bookings**.

**Smarter than random.** The next cell is **optional**. **Optuna** is a tuning library that *learns from each cake*: after a few tries it stops baking in the bad corner of the kitchen and searches near the good recipes. Same honesty rule: cross-validation does the tasting. (About a minute.)

In [ ]:
# OPTIONAL — Optuna: a tuner that learns from every trial where to look next (needs the pip install cell at the top)
try:
    import optuna; optuna.logging.set_verbosity(optuna.logging.WARNING)

    def taste(trial):
        # Optuna proposes a recipe, we answer with its cross-validation score
        recipe = dict(learning_rate=trial.suggest_float('learning_rate', 0.03, 0.3, log=True),
                      max_leaf_nodes=trial.suggest_int('max_leaf_nodes', 7, 127, log=True),
                      min_samples_leaf=trial.suggest_int('min_samples_leaf', 5, 100, log=True))
        return cross_val_score(new_boost(**recipe), X_tr, y_tr, cv=search.cv, scoring='roc_auc', n_jobs=-1).mean()

    study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=835))
    study.optimize(taste, n_trials=15)
    print(f'Optuna, 15 trials: best CV score {study.best_value:.4f}   {study.best_params}')
    print(f'random search, 12 trials: best CV score {search.best_score_:.4f}')
except ImportError:
    print('Optuna is not installed. Run the pip install cell near the top, then run this cell again. (Nothing later depends on it.)')

**Stop and notice:** Optuna lands in the **same neighborhood** as random search: small steps, fairly big trees, and a score within a hair. On a small problem like ours, random search is plenty. Optuna earns its keep when each model takes **hours** to train and every trial is precious.

## 5. The trap: the best of many guessers

Why are we so strict about **not** using the test bookings to pick the winner? Here is the reason, in one small experiment.

🎮 Play it first: in [Station 2 of the Trust Explorer](https://isom-835.vercel.app/explore/trust/#s2), your mission is to make a model that knows nothing look like a real one.

Imagine **200 "models" that know nothing**. Each one just writes down a **random number** for every booking. Pure guessing. We grade all 200 on a **small exam** of 200 test bookings, and we crown **the best one**.

In [ ]:
small_exam, fresh_bookings = y_te.values[:200], y_te.values[200:]

guessers = [np.random.default_rng(k).random(len(y_te)) for k in range(200)]     # 200 "models": random numbers, no knowledge at all
on_small_exam = [roc_auc_score(small_exam, g[:200]) for g in guessers]

winner = int(np.argmax(on_small_exam))
print(f'200 guessers on the small exam: the average scores {np.mean(on_small_exam):.2f}, the BEST one scores {max(on_small_exam):.2f}')
print(f'that same "best" guesser on {len(fresh_bookings):,} fresh bookings:  {roc_auc_score(fresh_bookings, guessers[winner][200:]):.2f}')

**Stop and notice:**
- Every guesser knows **nothing**, so the average score is **0.50**, a coin flip. But the **best of 200** scores about **0.62** on the small exam. It looks like it knows something!
- On fresh bookings, the "champion" falls right back to **0.50**. It was never good. It was **lucky**, and we picked it *because* it was lucky.

This is what happens whenever you try **many** things and keep the one with the **best score on the same exam**. The more you try, the luckier your winner looks. It is exactly what happened to teams who climbed the **public leaderboard** in our competition and then dropped on the **private** one.

So the rule has two parts:

1. **Choose** with cross-validation (the tasting kitchen).
2. **Open the test set once**, at the very end, only to **report** how good the chosen model is.

We have chosen. Now, and only now, we open the test set for the tuned model.

In [ ]:
tuned = search.best_estimator_                                # the winning recipe, already re-trained on all 24,000 training bookings
tuned_auc = roc_auc_score(y_te, tuned.predict_proba(X_te)[:, 1])

print(f"last week's champion (default dials):  test AUC {champion_auc:.3f}   ({champion.n_iter_} trees)")
print(f"tonight's tuned model:                 test AUC {tuned_auc:.3f}   ({tuned.n_iter_} trees)")

**Stop and notice:** about **0.947**, up from 0.945, and with fewer trees. The test set agrees with what cross-validation promised: a **small, real** gain. No surprise on exam day, because we chose honestly.

`tuned` is the model we take to the boss. On to the questions that numbers alone cannot answer.

### 🧠 Quick check

A teammate tried 60 different models. For each one he looked at the **test** AUC, and he kept the best: 0.951. He reports 0.951 to the client. What will probably happen when the model meets new bookings?

<details><summary>Click for the answer</summary>

It will probably score **lower** than 0.951. He used the test set 60 times to **choose**, so his winner is partly the **luckiest** of 60, like the best guesser. The test set is no longer a fair exam for it. He should have chosen with **cross-validation** and opened the test set **once**.
</details>

---
# Question 3 · "Why did it say that?"

## 6. A receipt for one booking: SHAP

The front-desk manager calls you: *"Your model says booking 438 has an **87% chance** of canceling. **Why?** I'm not giving away that room on a number I can't explain."*

A fair question. Our model is about 300 small trees added together. Nobody can read that. But we can ask for something a manager reads every day: **a receipt**.

- At a restaurant, the receipt starts at **$0** and lists what each dish **added**. The lines add up to the total.
- A **SHAP receipt** starts at **the average booking's score (about 37%)** and lists how many **points each clue added or took away** for *this* guest. The lines add up to the model's score for this guest.

How does it decide what each clue is "worth"? Think of **splitting a taxi fare fairly** among friends who got in at different stops: each pays for the part of the ride they caused. SHAP does that for clues. It asks: *"how much does the score move when this clue joins the others?"*, averaged over all the orders in which the clues could join. The idea comes from a Nobel-winning economist, Lloyd **Shap**ley, and the name is short for *SHapley Additive exPlanations*.

🎮 Play it first: in [Station 3 of the Trust Explorer](https://isom-835.vercel.app/explore/trust/#s3) **you** are the model. For eight real guests, call each clue "up" or "down", then reveal the real receipt.

The next cell computes the receipts for the first 500 test bookings. It takes **about half a minute**.

In [ ]:
# Needs the shap library, installed by the second cell at the top of the notebook.
try:
    import shap
except ModuleNotFoundError:
    raise ModuleNotFoundError('shap is not installed yet. Scroll to the top, run the  !pip install  cell, then run this cell again.') from None

explained = X_te.iloc[:500]                                   # the first 500 test bookings
typical = X_tr.sample(100, random_state=835)                  # 100 training bookings that stand for "the average booking"

explainer = shap.TreeExplainer(tuned, data=typical, model_output='probability')     # 'probability' = pushes measured in chance of canceling
sv = explainer(explained)
original = X_te_raw.iloc[:500][explained.columns]             # the same 500 bookings with their original words (e.g. "Non Refund")
sv.display_data = original.values                             # ...so the pictures show words instead of code numbers
chance = tuned.predict_proba(explained)[:, 1]                 # the model's score for each of the 500 bookings

def receipt(i, lines=6):
    # The receipt for booking i: where the score starts, the biggest pushes, and where it ends
    pushes = pd.DataFrame({'clue': explained.columns, 'this booking': original.iloc[i].values, 'push (points)': 100 * sv.values[i]})
    pushes = pushes.reindex(pushes['push (points)'].abs().sort_values(ascending=False).index)
    top, rest = pushes.head(lines), pushes.iloc[lines:]
    rows = [{'clue': 'START: the average booking', 'this booking': '', 'push (points)': 100 * sv.base_values[i]}]
    rows += top.to_dict('records')
    rows += [{'clue': f'all {len(rest)} other clues together', 'this booking': '', 'push (points)': rest['push (points)'].sum()}]
    rows += [{'clue': "TOTAL: the model's score", 'this booking': '', 'push (points)': 100 * (sv.base_values[i] + sv.values[i].sum())}]
    return pd.DataFrame(rows).round(1).set_index('clue')

print(f'receipts ready for {len(explained)} bookings')

Now the manager's booking. The cell prints the receipt as a table, then draws SHAP's standard picture of the same receipt, called a **waterfall**.

In [ ]:
# Needs the receipts cell above (Step 6)
i = 438                                                       # ← the manager's booking. Change this number to explain a different one.
print(f'Booking {i}: the model says {chance[i]:.0%} chance of canceling. What really happened: {"CANCELED" if y_te.iloc[i] == 1 else "showed up"}.\n')
print(receipt(i).to_string())
shap.plots.waterfall(sv[i], max_display=8)

**How to read the table:** start at the top with the **average booking (about 37 points)**. Each line is one clue: its value for this guest, and how many **points** it pushed the score **up (+)** or **down (−)**. The last line is the total: the model's score.

**How to read the picture (same information):** start at the **bottom** (`E[f(X)]`, the average). Each bar is one clue. **Red bars push the chance of canceling up, blue bars push it down.** Follow the staircase to the top: `f(x)`, the model's score for this guest. The picture writes the pushes as shares, so **+0.24 means +24 points**.

**Stop and notice:**
- You can now say it in one sentence to the manager: *"Every booking starts at the average, 37%. This one scores 87% mainly because the guest **booked 312 days ahead** (about +24 points), made **no special requests** (about +10) and came through an **online travel agency** (about +10). One clue pulls the other way: the guest already **changed the booking twice** (about −10), and people who bother to change a booking usually intend to come."*
- The receipt **adds up**. Nothing is hidden: every point of the score is assigned to a clue. (The total can differ from the model's score by a fraction of a point. That is rounding inside the method, nothing more.)
- The model was right about this guest: the booking was **canceled**.
- This is a **local** explanation: it is about **this one booking**. Another guest gets a different receipt.

🎮 You have seen a receipt before: Station 4 of the [House Price Explorer](https://isom-835.vercel.app/explore/regression/) showed where each dollar of a house price came from. Same idea, now for any model.

## 7. The whole book: which clues matter most?

One receipt explains **one** guest. The owner asks a bigger question: *"In general, what drives cancellations in your model?"*

Easy: we have 500 receipts. For each clue, take the **average size of its push** across all 500 (ignoring the + or − sign). A clue that moves scores by many points, guest after guest, **matters**. This is a **global** explanation.

In [ ]:
# Needs the receipts cell above (Step 6)
average_push = pd.Series(100 * np.abs(sv.values).mean(axis=0), index=explained.columns).sort_values()

fig, ax = plt.subplots(figsize=(8, 4.2))
average_push.tail(10).plot.barh(ax=ax, color='#5b6bd8')
for row, points in enumerate(average_push.tail(10)):
    ax.text(points + 0.15, row, f'{points:.1f}', va='center')
ax.set_xlabel('average push on the score, in points (up or down)'); ax.set_title('What drives the model? The 10 clues that move scores the most')
plt.show()

**Stop and notice:**
- Two clues lead: the guest's **country** (it moves a typical score by about 11 points) and the **deposit type** (about 9).
- Then a group of four, at 5 to 6 points each: **special requests**, the **travel agent**, the **market segment** (how they booked) and **lead time**.
- The other 20-odd clues each move the score only a little.

The bar chart says **how much** each clue matters. It does not say **in which direction**. For that, SHAP has a famous picture, the **beeswarm**. The next cell draws it for six number clues.

In [ ]:
# Needs the receipts cell above (Step 6)
six_clues = ['lead_time', 'total_of_special_requests', 'required_car_parking_spaces', 'previous_cancellations', 'adr', 'booking_changes']
columns = [list(explained.columns).index(c) for c in six_clues]
shap.plots.beeswarm(sv[:, columns], max_display=6)

**How to read it:** each **dot is one booking**. Each row is one clue.
- **Left or right** = the push. Right of the line: this clue pushed that booking's score **up** (toward "cancels"). Left: **down**.
- **Color** = the clue's value for that booking. **Red = a high value** (a long lead time, many requests). **Blue = a low value.**

**Stop and notice** (read one row at a time):
- **`lead_time`:** red dots on the **right**. Booking far ahead pushes the chance of canceling **up**. Blue dots (last-minute bookings) sit on the left.
- **`total_of_special_requests`:** the opposite. **Red dots on the left**: guests who ask for things (a crib, a high floor) are **committed**. Zero requests (blue) pushes risk up.
- **`required_car_parking_spaces`:** red dots far to the **left**. A guest who asked for a parking space almost never cancels.
- **`previous_cancellations`:** red on the **right**. People who canceled before tend to cancel again.
- **`booking_changes`:** red on the **left**. A guest who changes a booking is planning to come. (You saw this clue on the manager's receipt.)

(The bottom axis is in shares: 0.2 means 20 points.)

### ⚠️ The warning label: SHAP explains the model, not the world

Look again at the second-biggest clue in the bar chart: **deposit type**. Common sense says a guest who paid a **non-refundable** deposit will surely show up. The next cell checks what the data says.

In [ ]:
by_deposit = bookings.groupby('deposit_type')['is_canceled'].agg(bookings='size', cancel_rate='mean')
print(by_deposit.round(3).to_string())

**Stop and notice:** in this data, **non-refundable bookings were canceled about 99% of the time**. So the model learned: *"non-refundable → will cancel"*, and SHAP honestly reports a big **push up** for it.

Now imagine the owner reads that and says: *"Then let's **stop** asking for non-refundable deposits. Cancellations will drop!"*

**That would be a mistake.** The deposit does not *cause* the cancellation. Most likely these rows are **bulk bookings** by tour operators that were booked and canceled in blocks: a special kind of booking, not ordinary guests reacting to a deposit. The model found a **pattern in the records**. It knows nothing about what would happen if you **changed the policy**.

Two sentences to remember:

- SHAP answers: ***"why did the model give this score?"***
- SHAP does **not** answer: ***"what will happen if I change this?"*** For that you need an **experiment** (an A/B test), not an explanation.

### 🧠 Quick check

A receipt shows `country = PRT` with a push of **+12 points**. Which sentence is correct?

**(a)** "Being from Portugal makes a guest 12 points more likely to cancel."
**(b)** "For this booking, the model's score is 12 points higher because of what it learned about bookings from Portugal."

<details><summary>Click for the answer</summary>

**(b).** SHAP describes **the model's reasoning**, not a law of nature. In this data, bookings from Portugal include many bulk and agency bookings that were canceled, so the model leans on that clue. (a) claims a **cause** that nobody has shown. This also raises a **fairness** question, which is Step 9.
</details>

---
# Question 4 · "How sure is it?"

## 8. Teaching the model to say "I'm not sure": conformal prediction

So far our model answers every booking with total confidence, even the ones it is really guessing about. A good employee does better. A good employee says one of **three** things:

- ✅ *"This one **will stay**."*
- ❌ *"This one **will cancel**."*
- 🤔 *"**I'm not sure.** Somebody should look at this one."*

And we want a **promise** to go with it: *"The truth will be inside my answer **at least 90% of the time**."* ("Not sure" means "stay or cancel", so the truth is always inside that one.)

Think of a **weather forecaster** with a track record. If she says "90% sure" and, looking back over the year, she was right 90% of the time, you can **plan around her**. The method that gives a model this kind of honest track record is called **conformal prediction**. The recipe is short:

1. **Set aside a "track record" pile:** bookings the model has **never seen** (we take a quarter of the training bookings). Its real name is the **calibration set**.
2. For each of them, measure the model's **surprise**: *how little chance did it give to what really happened?* If a guest canceled and the model had said 95% cancel, surprise is small (0.05). If it had said 10%, surprise is big (0.90).
3. Find **the bar**: the level that 90% of the surprises stay under.
4. For a **new** booking, keep every answer that would surprise the model **less than the bar**. One answer kept: the model commits. **Both kept: "I'm not sure."**

That is all. No new model, no heavy math. **Two bookings make it click.** Say the bar comes out at **0.58** (the next cell will print the real one):

| | The model says | If the guest **cancels**, its surprise is… | If the guest **stays**, its surprise is… | Under the bar (0.58)? | The answer |
|---|---|---|---|---|---|
| **Booking A** | 80% cancel | 1 − 0.80 = **0.20** ✓ | 1 − 0.20 = **0.80** ✗ | only "cancel" | ❌ **"will cancel"** |
| **Booking B** | 50% cancel | 1 − 0.50 = **0.50** ✓ | 1 − 0.50 = **0.50** ✓ | both | 🤔 **"not sure"** |

Notice what happens: the bar turns into a **zone in the middle of the scores**. Any booking the model scores between about **42% and 58%** gets "not sure". Everything outside the zone gets a firm answer. (That is the amber zone you dragged open in the game.)

🎮 Play it first: in [Station 4 of the Trust Explorer](https://isom-835.vercel.app/explore/trust/#s4), your staff can check 600 bookings by hand. Find the strongest promise you can afford.

The next cell does it by hand.

In [ ]:
PROMISE = 0.90                                                # "the truth is inside my answer at least 90% of the time"

# 1. Set aside a track-record pile that the model never trains on
X_fit, X_track, y_fit, y_track = train_test_split(X_tr, y_tr, test_size=0.25, stratify=y_tr, random_state=835)
careful = new_boost(**search.best_params_).fit(X_fit, y_fit)  # the tuned recipe, trained on the other three quarters

# 2. Surprise on the track record = 1 − the chance the model gave to what really happened
chances = careful.predict_proba(X_track)                      # two columns: chance of "stays" (0), chance of "cancels" (1)
surprise = 1 - chances[np.arange(len(y_track)), y_track.values]

def answers(promise):
    bar = np.quantile(surprise, promise)                      # 3. the bar: this share of the track-record surprises stay under it
                                                              #    (libraries nudge the bar up a hair when the pile is small)
    new = careful.predict_proba(X_te)                         # 4. new bookings: keep every answer that surprises less than the bar
    keep_stay, keep_cancel = (1 - new[:, 0]) <= bar, (1 - new[:, 1]) <= bar
    answer = np.select([keep_stay & keep_cancel, keep_stay, keep_cancel], ['not sure', 'will stay', 'will cancel'], default='no answer')
    truth_inside = np.where(y_te.values == 1, keep_cancel, keep_stay)
    return pd.Series(answer, index=y_te.index), truth_inside

answer, truth_inside = answers(PROMISE)
bar = np.quantile(surprise, PROMISE)
print(f'the bar: {bar:.2f}   →   any booking the model scores between {1 - bar:.0%} and {bar:.0%} gets "not sure"\n')
print(f'the promise: {PROMISE:.0%}   →   kept on {len(y_te):,} new bookings: {truth_inside.mean():.1%}\n')
print((answer.value_counts(normalize=True) * 100).round(1).astype(str).add('%').to_string())


def right_when_it_commits(answer):
    committed = answer.isin(['will stay', 'will cancel'])
    return (answer[committed].map({'will cancel': 1, 'will stay': 0}) == y_te[committed]).mean()

print(f'\nwhen the model commits to one answer, it is right {right_when_it_commits(answer):.1%} of the time')

**Stop and notice:**
- **The promise was kept.** We asked for 90%, and on 6,000 bookings the model has never seen, the truth was inside its answer about **90%** of the time. Not by luck: this is what the method **guarantees**, for *any* model, as long as new bookings look like the track-record bookings.
- The model **commits** on more than **9 bookings out of 10**, and says **"not sure"** on the rest (about 1 in 15).
- That "not sure" pile is **gold** for a business. It is the short list for a **human**: call the guest, ask for a deposit, look closer. The model handles the easy cases. People handle the hard ones.

What if the boss wants to be **more** sure? The next cell raises the promise.

In [ ]:
rows = []
for promise in [0.90, 0.95, 0.99]:
    answer_p, inside_p = answers(promise)
    rows.append({'the promise': f'{promise:.0%}', 'kept': f'{inside_p.mean():.1%}',
                 'says "not sure"': f"{(answer_p == 'not sure').mean():.0%}", 'commits': f"{(answer_p != 'not sure').mean():.0%}",
                 'right when it commits': f'{right_when_it_commits(answer_p):.1%}'})
print(pd.DataFrame(rows).set_index('the promise').to_string())

**Stop and notice:**
- Every promise is **kept** (the `kept` column matches the promise, give or take half a point: 6,000 bookings is a sample, not infinity).
- Certainty has a **price**: at 90% the model says "not sure" about **1 booking in 15**. At 95%, about **1 in 5**. At 99%, almost **half**.
- And it has a **reward**: look at the last column. The more often the model is allowed to say "not sure", the more you can **trust it when it does commit**: from about 89% right, to 93%, to almost 99%.
- This is a **business** decision, not a math one. *How much is a wrong call worth to us, and how many bookings can our staff review by hand?* The model gives you the menu. You choose.

**For numbers, the same trick gives a range.** In Session 6 the model said "this house is worth $214,000". With conformal prediction it says "**between $190,000 and $240,000**, and I keep that promise 90% of the time". 🎮 Try it in the [Conformal Explorer](https://isom-835.vercel.app/explore/conformal/): set the promise, watch the band widen and narrow, and test whether the promise holds.

In practice you do not write these lines yourself. The library is called **MAPIE** (`pip install mapie`), and it works with any scikit-learn model.

### 🧠 Quick check

Your boss says: *"I want the model to be 99.9% sure and never say 'not sure'."* What do you tell her?

<details><summary>Click for the answer</summary>

**You cannot have both.** The model's knowledge is fixed. If you ask for a stronger promise, it has to say "not sure" more often: that is the only honest way to be right more often. To get **more certainty and more commitment at once**, you need a **better model or better data** (new clues), not a bigger demand.
</details>

---
# Question 5 · "Does it work for everyone?"

## 9. Slices: the average can hide a group

*"The average grade in the class is a B+."* Sounds fine. But what if one group of students is failing and the rest are getting A's? **An average can hide a group that is badly served.**

Same with a model. To look behind the average, we **slice**: we compute the model's mistakes **separately for each group** of guests. First, a reminder (Sessions 4 and 5) that a yes/no model can be wrong in **two** ways. We flag a booking as "will cancel" when its score is 50% or higher.

| The mistake | In plain words | What it costs the hotel | Real name |
|---|---|---|---|
| **Missed cancel** | we said "will stay", the guest **canceled** | an **empty room** | false negative |
| **False alarm** | we said "will cancel", the guest **came** | we may have **given the room away**, or treated a good guest with suspicion | false positive |

The next cell computes both mistake rates for all guests, and then for three ways of slicing them.

In [ ]:
flagged = (tuned.predict_proba(X_te)[:, 1] >= 0.5).astype(int)        # 1 = the model says "will cancel"
canceled = y_te.values

def mistakes(group):
    # The model's two kinds of mistakes inside one group of test bookings (group = True/False for each booking)
    real, said = canceled[group], flagged[group]
    return {'bookings': int(group.sum()),
            'cancel rate': real.mean(),
            'missed cancels': ((real == 1) & (said == 0)).sum() / (real == 1).sum(),     # of the guests who canceled, how many did we miss?
            'false alarms':   ((real == 0) & (said == 1)).sum() / (real == 0).sum()}     # of the guests who came, how many did we wrongly flag?

def slice_by(labels, at_least=200):
    labels = pd.Series(np.asarray(labels))
    groups = [g for g, n in labels.value_counts().items() if n >= at_least]              # skip tiny groups: too few bookings to judge
    return pd.DataFrame({g: mistakes((labels == g).values) for g in groups}).T.astype({'bookings': int}).round(3)

everyone = pd.DataFrame({'ALL GUESTS': mistakes(np.ones(len(canceled), dtype=bool))}).T.astype({'bookings': int}).round(3)
by_hotel = slice_by(X_te_raw['hotel'])
by_channel = slice_by(X_te_raw['market_segment'])
by_country = slice_by(np.where(X_te_raw['country'] == 'PRT', 'Portugal', 'other countries'))

for title, table in [('everyone', everyone), ('by hotel', by_hotel), ('by how they booked (market segment)', by_channel), ('by country', by_country)]:
    print(f'— {title} —'); print(table.to_string(), '\n')

**How to read it:** each row is a group of test bookings. `missed cancels`: of the guests in that group who **canceled**, the share the model **failed to flag**. `false alarms`: of the guests who **came**, the share the model **wrongly flagged**. Lower is better for both.

The next cell draws the two slices that matter most.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 3.8))
for ax, table, title in [(axes[0], by_channel, 'By how they booked'), (axes[1], by_country, 'By country')]:
    (table[['missed cancels', 'false alarms']] * 100).plot.barh(ax=ax, color=['#d64545', '#f0a030'], width=0.75)
    ax.set_xlabel('% of bookings in that group'); ax.set_title(title); ax.invert_yaxis()
    for bars in ax.containers:
        ax.bar_label(bars, fmt='%.0f%%', padding=3)
    ax.set_xlim(0, 100 * table[['missed cancels', 'false alarms']].max().max() * 1.18)
plt.tight_layout(); plt.show()

**Stop and notice:**
- **Overall** the model misses about **1 cancel in 5** and raises a false alarm for about **1 arriving guest in 12**.
- **By how they booked:** the gaps are large.
  - **Groups** and **offline travel agents:** the model is excellent (it misses fewer than 1 cancel in 10).
  - **Online travel agencies** (Booking.com, Expedia…): the **biggest** segment, and the **most false alarms** (1 arriving guest in 7), with 3 cancels in 10 missed.
  - **Direct** bookings: it **misses 6 cancels in 10**. Those guests rarely cancel, so the model almost never dares to flag one.
  - The overall average was hiding all of this.
- **By country:** the model treats the two groups in **opposite** ways. Guests booking **from Portugal** get **more false alarms** (about 11 or 12 in 100, against 7): among guests who *did show up*, a Portuguese guest was at least one and a half times as likely to have been flagged as a canceler. For guests from **other countries** the model **misses a third of the cancels**. It has learned "Portugal → cancels" and it leans on it (remember Step 7: country was the number-one clue).
- If "flagged" means *"we give your room to someone else"* or *"we ask you for a deposit"*, then guests from one country are being treated **worse for the same behavior**. That is a **fairness** problem, and it would not appear in any overall score.

**What do you do about it?** There is no button. These are **management** decisions:
- **Know it.** You cannot manage a gap you never measured. Slicing is the measuring.
- **Decide what the model is allowed to do.** "Send a reminder email to flagged bookings" harms nobody. "Cancel flagged bookings" does. Use the model for the first, not the second.
- **Route the weak slices to people.** Online-agency bookings in the "not sure" pile (Step 8) go to a human.
- **Write it down**, so the next person knows. That is the last step.

In lending, hiring, insurance and health care, checking a model's mistakes by group is **not optional**: regulators ask for it (in Europe, the EU AI Act calls these "high-risk" uses).

## 10. The label on the box: a model card

Every food package has a **nutrition label**. Every medicine has a leaflet: what it is for, how to use it, the side effects. A model deserves the same. It is called a **model card**: **one page** that tells the truth about the model, written for the person who will use it, not for the person who built it.

Everything on it comes from what you did tonight:

> ### MODEL CARD · Hotel cancellation risk, version 1
>
> **What it is for.** Ranking bookings by their chance of cancellation, to plan overbooking and to decide which guests get a reminder call.
> **What it is NOT for.** Canceling or refusing an individual guest's booking without a person looking at it.
>
> **The data.** 30,000 bookings from two hotels in Portugal, arrivals July 2015 to August 2017. Three leaking columns removed (Session 7).
>
> **The model.** Gradient boosting with early stopping. Dials tuned with random search and cross-validation (Step 4).
>
> **How good is it?** *(Question 1 and 2)*
> - AUC **0.947** on 6,000 shuffled test bookings. Cross-validation: about **0.94, give or take 0.003**.
> - ⚠️ On the exam that looks like the real job (learn 2015–16, predict 2017): about **0.88**. **Expect this one in production.**
>
> **Why does it say what it says?** *(Question 3)* Mostly six clues: country, deposit type, special requests, travel agent, market segment and lead time. It explains its own scores. It does **not** tell you what happens if you change a policy.
>
> **How sure is it?** *(Question 4)* With a 90% promise it commits on more than 9 bookings in 10 and says "not sure" on the rest. Send those to a person.
>
> **Does it work for everyone?** *(Question 5)* Most false alarms on **online travel agency** bookings. Misses most cancels among **direct** bookings. Raises **more false alarms for guests booking from Portugal**, and misses more cancels from other countries.
>
> **Watch after launch.** Re-check the AUC every month once real outcomes arrive. Retrain when it falls, or when the mix of bookings changes.
>
> **Owner:** \_\_\_\_\_\_\_\_ · **Next review:** \_\_\_\_\_\_\_\_

**Stop and notice:** a model card is not marketing. The two ⚠️ lines and the "weakest on" lines are the **most valuable** lines on the page. They are what protects the hotel, and you, when something goes wrong.

Your **final project** includes a half-page model card. You now have a template.

## 11. Your turn

Each question has a starter cell. Change the line marked `# ← your turn`, run it, then answer in words. Work with a neighbor if you like.

1. **More exams.** Run cross-validation with 10 parts instead of 5. Does the average change? Does the spread?
2. **Explain another booking.** Pick a booking, print its receipt, and write the sentence you would say to the front-desk manager.
3. **Change the promise.** Try a promise of 0.97, and then 0.80. What happens to the "not sure" pile?
4. **Slice it your way.** Check the model's mistakes by `customer_type`, or by another column you are curious about.

**Q1 · More exams.** Change `n_splits` to 10.

In [ ]:
# Q1 · cross-validation with more parts
n_splits = 5                # ← your turn: change to 10
exams = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=835)
scores = cross_val_score(new_boost(), X_tr, y_tr, cv=exams, scoring='roc_auc')
print('the scores:', '  '.join(f'{s:.3f}' for s in scores))
print(f'{n_splits} exams: average {scores.mean():.3f}, give or take {scores.std():.3f} · lowest {scores.min():.3f}, highest {scores.max():.3f}')

Your answer to Q1 (double-click to edit): *with 10 exams the average is \_\_\_ (with 5 it was \_\_\_). The single scores spread more / less, because each exam now has fewer / more bookings…* (Hint: how many bookings are in each exam when you cut 24,000 into 10 parts? Is a small exam more or less lucky than a big one?)

**Q2 · Explain another booking.** Pick any number from 0 to 499. Three interesting ones: **2** (the model is very sure), **7** (the model is torn), **103** (the model is very sure, and wrong).

In [ ]:
# Q2 · a receipt for a booking you choose (needs the SHAP cells in Step 6)
i = 7                       # ← your turn: pick a booking from 0 to 499
print(f'Booking {i}: the model says {chance[i]:.0%} chance of canceling. What really happened: {"CANCELED" if y_te.iloc[i] == 1 else "showed up"}.\n')
print(receipt(i).to_string())
shap.plots.waterfall(sv[i], max_display=8)

Your answer to Q2: *"This booking starts at the average, 37%. The model gives it \_\_\_% mainly because \_\_\_ and \_\_\_. What pulls the other way is \_\_\_."* Was the model right about this guest?

**Q3 · Change the promise.**

In [ ]:
# Q3 · a different promise
promise = 0.90              # ← your turn: try 0.97, then 0.80
answer_q3, inside_q3 = answers(promise)
print(f'the promise: {promise:.0%}   →   kept: {inside_q3.mean():.1%}\n')
print((answer_q3.value_counts(normalize=True) * 100).round(1).astype(str).add('%').to_string())

Your answer to Q3: *at 0.97 the model says "not sure" on \_\_\_% of bookings. At 0.80 something new appears: \_\_\_. I think that happens because…* (Hint: at 0.80 you are asking for **less** than the model can deliver. It can be right about 87% of the time by itself. To be right only 80% of the time, it has to **refuse to answer** some bookings: "no answer".)

**Q4 · Slice it your way.**

In [ ]:
# Q4 · the model's mistakes by a column you choose
column = 'customer_type'    # ← your turn: also try 'deposit_type', 'distribution_channel', 'is_repeated_guest', 'arrival_date_year'
print(slice_by(X_te_raw[column]).to_string())

Your answer to Q4: *the group the model serves worst is \_\_\_ (missed cancels \_\_\_, false alarms \_\_\_). If I were the manager, I would…*

## What we learned tonight

Five questions from the boss, five answers with evidence:

- **"Is the score real?"** One exam is partly luck. Use **cross-validation** (five exams, report the average and the spread), and make the exam **look like the real job**: if the model will predict the future, test it on the future. Our 0.945 was closer to **0.94** on average, and to **0.88** on next year's bookings.
- **"Can we make it better?"** **Tuning** turns the dials (hyperparameters). **Choose with cross-validation, open the test set once.** If you pick the best of many on the same exam, you pick the luckiest. Tuning is the last few percent.
- **"Why did it say that?"** **SHAP** gives every prediction a **receipt**: start at the average, add each clue's push, arrive at the score. It explains **the model, not the world**.
- **"How sure is it?"** **Conformal prediction** lets the model say **"not sure"** and keeps an honest promise ("right at least 90% of the time"). The "not sure" pile goes to a human.
- **"Does it work for everyone?"** **Slice** the mistakes by group. The average hides the groups a model serves badly. Then write it all on **one page: the model card**.

### Tonight's words, and their real names

| What we said | The real name | What it means |
|---|---|---|
| five exams | **k-fold cross-validation** | every part of the training data takes one turn as the exam |
| a part | **fold** | one of the k equal pieces |
| a dial | **hyperparameter** | a setting you choose before training |
| turning the dials | **tuning** (random search, Optuna) | trying settings and keeping the best, judged by cross-validation |
| the receipt | **SHAP values** (waterfall plot) | each clue's push on one prediction, adding up to the score |
| one booking / the whole book | **local / global explanation** | why this prediction / what drives the model overall |
| the track-record pile | **calibration set** | held-back data used to measure how surprised the model gets |
| "will stay", "will cancel", "not sure" | **prediction set** | the answers the model cannot rule out |
| the promise | **coverage** | how often the truth is inside the model's answer |
| missed cancel / false alarm | **false negative / false positive** | the two ways a yes/no model can be wrong |
| a group of guests | **slice** | a part of the data you check separately |
| the label on the box | **model card** | one honest page about what the model is for and where it fails |

### What's due

- **Tonight, Mon Nov 16, 11:59 PM** · HW3 on Canvas (thresholds from Session 5 + Ames regression from Session 6).
- **Mon Nov 23, 11:59 PM** · Project Milestone 2: your data loaded, split before anything is fitted, a naive baseline, and a first model scored with the metric your decision needs.
- **Mon Nov 30, 11:59 PM** · Project Milestone 3: a **tuned** model (dials chosen inside cross-validation), **SHAP** receipts for three cases, mistakes by the **slices** that matter, and a half-page **model card**. Tonight's notebook has every piece: copy the pattern onto your own data.

**Next Monday (Session 10):** learning **without** answers. Clustering customers into segments, seeing many columns at once, and spotting the transactions that don't belong.